# HotBob Modal Notebook Runner

Use this notebook in Modal Notebooks to run HotBob LLM-memory experiments on cloud GPU compute with persistent data on a Modal Volume.

In the Modal notebook sidebar, target a **1x NVIDIA T4** kernel and attach a Volume mounted at `/mnt/hotbob`. Modal mounts attached Volumes under `/mnt`, and files outside a Volume are ephemeral across kernel restarts.

In [ ]:
import subprocess
import sys

print(sys.version)
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("No NVIDIA GPU visible. In Modal, set the notebook kernel GPU to 1x T4.")

## Configure the run

Set `REPO_URL` to your linked GitHub repository. `BRANCH` can be `master`, your current experiment branch, or any ref that exists on GitHub.

The repo, generated datasets, checkpoints, Hugging Face cache, Torch cache, and uv cache live under `/mnt/hotbob` so they survive Modal kernel restarts.

In [ ]:
import os
from pathlib import Path

VOLUME_ROOT = "/mnt/hotbob"
Path(VOLUME_ROOT).mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/Popidge/hotbob.git"
# Use a branch/ref that contains the code you want Modal to run.
# The architecture comparison module lives on feature/memory-architecture-comparison.
BRANCH = "feature/memory-architecture-comparison"
PROJECT_DIR = f"{VOLUME_ROOT}/repo"

os.environ["HF_HOME"] = f"{VOLUME_ROOT}/hf-cache"
os.environ["TRANSFORMERS_CACHE"] = f"{VOLUME_ROOT}/hf-cache/transformers"
os.environ["TORCH_HOME"] = f"{VOLUME_ROOT}/torch-cache"
os.environ["UV_CACHE_DIR"] = f"{VOLUME_ROOT}/uv-cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
for path in [
    os.environ["HF_HOME"],
    os.environ["TRANSFORMERS_CACHE"],
    os.environ["TORCH_HOME"],
    os.environ["UV_CACHE_DIR"],
]:
    Path(path).mkdir(parents=True, exist_ok=True)

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
SEED = 41

# Start small to confirm the notebook is healthy, then scale up.
TRAIN_N = 10_000
EVAL_N = 1_000
TRAIN_STEPS = 1_000
BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1

INTEGRATION_MODE = "prefix"  # prefix, attention_q, attention_o, attention_qo
MEMORY_STATE_MODE = "shared"  # shared, by_type
ATTENTION_PATCH_LAYERS = "last4"  # all, last4, or comma-separated layer indexes
CORRECTION_RANK = 16
WRITE_LOSS_WEIGHT = 0.2

TRAIN_TRACES = "data/llm_train_colab.jsonl"
EVAL_TRACES = "data/llm_eval_colab.jsonl"
CHECKPOINT = f"runs/qwen_memory/colab_{INTEGRATION_MODE}_{TRAIN_N}_{TRAIN_STEPS}.pt"

# Optional hotbob.llm.architecture_compare settings.
# Variants are mode[:layers] or mode:by_type[:layers]. Set to [] to use the module defaults.
RUN_ARCHITECTURE_COMPARISON = False
ARCHITECTURE_VARIANTS = [
    "prefix",
    "attention_q:last4",
    "attention_o:last4",
    "attention_qo:last4",
]
ARCHITECTURE_EVAL_MODES = ["teacher_forced"]
ARCHITECTURE_DECODE_STRATEGY = "score_answers"
ARCHITECTURE_FREEZE_BASE = True
ARCHITECTURE_RUN_DIR = "runs/qwen_memory/architecture_compare"
ARCHITECTURE_REPORT_NAME = "comparison.json"
ARCHITECTURE_LOSS_BUCKET_SIZE = 1000

## Optional Hugging Face token

Public Qwen downloads usually work without a token. If you need one, attach a Modal Secret that exposes `HF_TOKEN`; the notebook will also use an existing environment variable if one is already present.

In [ ]:
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("HF_TOKEN is available from the environment.")
else:
    print("No HF_TOKEN found; continuing without one.")

## Clone or update the repo on the Volume

In [ ]:
import os
import subprocess
from pathlib import Path

repo_path = Path(PROJECT_DIR)
if repo_path.exists():
    os.chdir(PROJECT_DIR)
    subprocess.run(["git", "fetch", "--all", "--prune"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    os.chdir(PROJECT_DIR)

subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
subprocess.run(["git", "status", "--short", "--branch"], check=True)
print(f"Working directory: {Path.cwd()}")

## Install dependencies

This uses the repo's `uv.lock` and PyTorch CUDA index from `pyproject.toml`. The uv cache is on the Modal Volume.

In [ ]:
!python -m pip install -q uv
!uv sync

## Quick verification

In [ ]:
import subprocess

import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

subprocess.run(["uv", "run", "pytest"], check=True)

## Generate synthetic LLM traces

In [ ]:
!uv run python -m hotbob.llm.generate_data \
  --train-out {TRAIN_TRACES} \
  --eval-out {EVAL_TRACES} \
  --train-n {TRAIN_N} \
  --eval-n {EVAL_N} \
  --seed {SEED}

## Train a single memory adapter

In [ ]:
!uv run python -m hotbob.llm.train \
  --model {MODEL} \
  --traces {TRAIN_TRACES} \
  --steps {TRAIN_STEPS} \
  --batch-size {BATCH_SIZE} \
  --integration-mode {INTEGRATION_MODE} \
  --memory-state-mode {MEMORY_STATE_MODE} \
  --correction-rank {CORRECTION_RANK} \
  --attention-patch-layers {ATTENTION_PATCH_LAYERS} \
  --write-loss-weight {WRITE_LOSS_WEIGHT} \
  --freeze-base \
  --out {CHECKPOINT}

## Evaluate

In [ ]:
!uv run python -m hotbob.llm.evaluate \
  --checkpoint {CHECKPOINT} \
  --traces {EVAL_TRACES} \
  --model {MODEL} \
  --mode all \
  --batch-size {EVAL_BATCH_SIZE} \
  --decode-strategy score_answers

## Optional architecture comparison

Run this when you want matched prefix vs attention-correction variants from the same data and step count.

In [ ]:
import subprocess
import importlib.util

if RUN_ARCHITECTURE_COMPARISON:
    if importlib.util.find_spec("hotbob.llm.architecture_compare") is None:
        raise RuntimeError(
            "hotbob.llm.architecture_compare is not available in this checkout. "
            "Set BRANCH to a ref that contains src/hotbob/llm/architecture_compare.py, "
            "then rerun the clone/update and install cells."
        )

    cmd = [
        "uv", "run", "python", "-m", "hotbob.llm.architecture_compare",
        "--model", MODEL,
        "--train-traces", TRAIN_TRACES,
        "--eval-traces", EVAL_TRACES,
        "--steps", str(TRAIN_STEPS),
        "--batch-size", str(BATCH_SIZE),
        "--eval-batch-size", str(EVAL_BATCH_SIZE),
        "--correction-rank", str(CORRECTION_RANK),
        "--write-loss-weight", str(WRITE_LOSS_WEIGHT),
        "--decode-strategy", ARCHITECTURE_DECODE_STRATEGY,
        "--run-dir", ARCHITECTURE_RUN_DIR,
        "--report-name", ARCHITECTURE_REPORT_NAME,
        "--loss-bucket-size", str(ARCHITECTURE_LOSS_BUCKET_SIZE),
    ]
    cmd.append("--freeze-base" if ARCHITECTURE_FREEZE_BASE else "--no-freeze-base")

    if ARCHITECTURE_EVAL_MODES:
        cmd.append("--eval-modes")
        cmd.extend(ARCHITECTURE_EVAL_MODES)

    for variant in ARCHITECTURE_VARIANTS:
        cmd.extend(["--variant", variant])

    print("Running:", " ".join(cmd))
    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if result.returncode:
        raise SystemExit(result.returncode)

## Inspect and persist artifacts

Checkpoints, reports, datasets, and caches are under `/mnt/hotbob`. Run this cell after long runs to flush pending writes to the Modal Volume.

In [ ]:
import subprocess

!find runs -maxdepth 5 -type f | sort

subprocess.run(["sync", VOLUME_ROOT], check=False)
print(f"Volume root: {VOLUME_ROOT}")
print(f"Main checkpoint: {CHECKPOINT}")
print(f"Architecture report: {ARCHITECTURE_RUN_DIR}/{ARCHITECTURE_REPORT_NAME}")